In [ ]:
import json
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import BertTokenizer, BertModel, XLMRobertaModel, DistilBertModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()

        # BERT model
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        # XLM-Roberta model
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        # DistilBERT model
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Dropout layer
        self.dropout = nn.Dropout(0.1)
        # Linear layer
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state

        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        concatenated = self.dropout(concatenated)

        out = self.fc(concatenated[:, 0, :])
        return out

def load_model():
    model = CustomBERTModel(num_labels=2).to(device)
    #model.load_state_dict(torch.load("best_model.pth"))
    model.load_state_dict(torch.load("best_model.pth", map_location=device))

    model.eval()
    return model

def predict(text_list):
    tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    encodings = tokenizer(text_list, truncation=True, padding='max_length', max_length=256, return_tensors="pt")
    input_ids = encodings["input_ids"].to(device)
    
    model = load_model()
    
    with torch.no_grad():
        outputs = model(input_ids)
    predictions = torch.argmax(outputs, dim=1).cpu().numpy().tolist()
    return predictions

model = load_model()
model.eval()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

files = ['test_df','dev_en_news', 'dev_en_reviews', 'dev_en_twitter', 'dev_nl_news', 'dev_nl_reviews', 'dev_nl_twitter']
results = {}

for file in files:
    # Read CSV
    df = pd.read_csv(f"{file}.csv")
    texts = df['text'].tolist()  
    true_labels = df['label'].tolist()  

    # Predict for all rows
    predicted_labels = [predict(text) for text in texts]

    # Calculate metrics
    accuracy = accuracy_score(true_labels, predicted_labels)
    f1 = f1_score(true_labels, predicted_labels, average='macro')

    results[file] = {'Accuracy': accuracy, 'F1 Score': f1}

# Print the results
for file, metrics in results.items():
    print(f"File: {file}")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(f"F1 Score: {metrics['F1 Score']:.4f}")
    print("----------")

In [1]:
import json
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import (BertTokenizer, BertModel, 
                          XLMRobertaModel, XLMRobertaTokenizer, 
                          DistilBertModel, DistilBertTokenizer)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()

        # BERT model
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        # XLM-Roberta model
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        # DistilBERT model
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        
        # Dropout layer
        self.dropout = nn.Dropout(0.1)
        # Linear layer
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, input_ids):
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state

        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        concatenated = self.dropout(concatenated)

        out = self.fc(concatenated[:, 0, :])
        return out

def load_model():
    model = CustomBERTModel(num_labels=2).to(device)
    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    model.eval()
    return model

def predict(text_list, model, tokenizer):
    encodings = tokenizer(text_list, truncation=True, padding='max_length', max_length=256, return_tensors="pt")
    input_ids = encodings["input_ids"].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids)
    predictions = torch.argmax(outputs, dim=1).cpu().numpy().tolist()
    return predictions

model = load_model()
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')  # TODO: Consider other tokenizers too

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

files = ['test_df','dev_en_news', 'dev_en_reviews', 'dev_en_twitter', 'dev_nl_news', 'dev_nl_reviews', 'dev_nl_twitter']

results = {}
BATCH_SIZE = 32

for file in files:
    df = pd.read_csv(f"{file}.csv")
    texts = df['text'].tolist()  
    true_labels = df['label'].tolist()  
    predicted_labels = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i+BATCH_SIZE]
        batch_predictions = predict(batch_texts, model, tokenizer)
        predicted_labels.extend(batch_predictions)

    accuracy = accuracy_score(true_labels, predicted_labels)
    f1 = f1_score(true_labels, predicted_labels, average='macro')
    results[file] = {'Accuracy': accuracy, 'F1 Score': f1}

for file, metrics in results.items():
    print(f"File: {file}")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(f"F1 Score: {metrics['F1 Score']:.4f}")
    print("----------")

/home/hmohammadi/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


File: test_df
Accuracy: 0.5333
F1 Score: 0.4034
----------
File: dev_en_news
Accuracy: 0.6650
F1 Score: 0.6518
----------
File: dev_en_reviews
Accuracy: 0.5050
F1 Score: 0.3443
----------
File: dev_en_twitter
Accuracy: 0.5450
F1 Score: 0.4705
----------
File: dev_nl_news
Accuracy: 0.5800
F1 Score: 0.5251
----------
File: dev_nl_reviews
Accuracy: 0.5200
F1 Score: 0.3981
----------
File: dev_nl_twitter
Accuracy: 0.5050
F1 Score: 0.3443
----------
